# LieDetector finetune v3: 实时域强化
数据: lie-detector-live-v1 (29.6fps 实时采集带遥测GT, 50轮) + lie-detector-new-v1 (旧采集)
目标: 压制实时闭环隐身期 FP/漏检, 提升 hidden 命中

In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO
import shutil, glob, os, yaml

In [ ]:
# 挂载数据集 (自动发现; 符号链接合并 — 不再物理拷贝 2 万文件, 上版拷贝浪费 2h GPU)
# v6 修复: 链接必须落到 <split>/images 与 <split>/labels 两个子目录 (v5 建到了 MERGE/train 平级导致 "No images found")
print("input mounts:", os.listdir("/kaggle/input"))
def find_mount(slug):
    for root, dirs, files in os.walk("/kaggle/input"):
        if os.path.basename(root) == slug:
            return root
    raise RuntimeError(f"mount not found for {slug}; tree={os.listdir('/kaggle/input')}")

DS_LIVE = find_mount("lie-detector-live-v1")
DS_OLD = find_mount("lie-detector-new-v1")
print("DS_LIVE:", DS_LIVE, os.listdir(DS_LIVE)[:8])
print("DS_OLD:", DS_OLD)
MERGE = "/kaggle/working/ds_merged"
for d in (f"{MERGE}/train/images", f"{MERGE}/train/labels",
          f"{MERGE}/val/images", f"{MERGE}/val/labels"):
    os.makedirs(d, exist_ok=True)

def link(fp, img_dir, lbl_dir, name=None):
    base = os.path.splitext(os.path.basename(fp))[0]
    ext = os.path.splitext(fp)[1]
    stem = name or base
    dst = os.path.join(img_dir, stem + ext)
    if not os.path.lexists(dst):
        os.symlink(fp, dst)
    lfp = os.path.join(os.path.dirname(os.path.dirname(fp)), "labels", base + ".txt")
    if os.path.exists(lfp):
        ldst = os.path.join(lbl_dir, stem + ".txt")
        if not os.path.lexists(ldst):
            os.symlink(lfp, ldst)

n_old = 0
for split in ("train", "val"):
    for fp in glob.glob(f"{DS_OLD}/**/{split}/images/*", recursive=True):
        link(fp, f"{MERGE}/train/images", f"{MERGE}/train/labels", "old_" + os.path.splitext(os.path.basename(fp))[0])
        n_old += 1

yamls = glob.glob(f"{DS_LIVE}/**/data.yaml", recursive=True)
assert yamls, f"no data.yaml under {DS_LIVE}"
live_root = os.path.dirname(yamls[0])
print("live_root:", live_root, os.listdir(live_root)[:8])
n_live_tr = 0
for fp in glob.glob(f"{live_root}/train/images/*"):
    link(fp, f"{MERGE}/train/images", f"{MERGE}/train/labels"); n_live_tr += 1
n_live_va = 0
for fp in glob.glob(f"{live_root}/val/images/*"):
    link(fp, f"{MERGE}/val/images", f"{MERGE}/val/labels"); n_live_va += 1
print(f"old {n_old} + live train {n_live_tr} / live val {n_live_va}")

# v6: 训练前硬校验 — 合并结果必须真实存在且非空, 防止再次空跑 20 分钟
for sub, min_n in (("train", 5000), ("val", 500)):
    n_img = len(glob.glob(f"{MERGE}/{sub}/images/*"))
    n_lbl = len(glob.glob(f"{MERGE}/{sub}/labels/*"))
    print(f"merged {sub}: {n_img} images / {n_lbl} labels")
    assert n_img >= min_n, f"merged {sub}/images only {n_img} < {min_n}"
    assert n_lbl > 0, f"merged {sub}/labels empty"

work_yaml = "/kaggle/working/data.yaml"
with open(yamls[0]) as f:
    cfg = yaml.safe_load(f)
cfg["path"] = MERGE
with open(work_yaml, "w") as f:
    yaml.safe_dump(cfg, f)
print(cfg)
print("BASE_CKPT:", os.path.join(DS_LIVE, "best.pt"),
      os.path.exists(os.path.join(DS_LIVE, "best.pt")))


In [ ]:
BASE = os.path.join(DS_LIVE, "best.pt")      # 挂载布局自适应, 不写死路径
assert os.path.exists(BASE), f"base weights missing at {BASE}"
model = YOLO(BASE)
results = model.train(
    data=work_yaml,
    epochs=40,
    imgsz=640,
    batch=32,
    lr0=0.002,
    patience=15,
    project="/kaggle/working/runs",
    name="train",
)

In [ ]:
metrics = model.val()
print(f"mAP50-95: {metrics.box.map:.4f}, mAP50: {metrics.box.map50:.4f}")
names = metrics.names
for i, ci in enumerate(metrics.box.ap_class_index):
    print(f"{names[ci]:<12} mAP50={metrics.box.ap50[i]:.4f} mAP50-95={metrics.box.ap[i]:.4f}")
best = "/kaggle/working/runs/train/weights/best.pt"
shutil.copy(best, "/kaggle/working/best_ft3.pt")
print("done -> best_ft3.pt")